In [1]:
'''Setup entire script and define raw raster download'''
from utils.util import *
import geopandas as gpd
import os
import sys
import time
import pandas as pd
import rasterio
import rioxarray
import socket
import traceback

def gatherRawRasters(dataset, year, city, aoi_geodf):
    print(f'Gathering {dataset} for {year} in {city}.')
    if dataset == 'srtm_v3' and year != 2014:
        return
    bandNames = {'B2', 'B3', 'B4', 'B5', 'B6', 'ST_B10', 'QA_PIXEL'}
    for month in range(1, 13):
        #Search for scenes
        print("Starting month", month)
        clear_folder(unprocessed_dir)
        search_payload = createSceneSearchPayload(dataset, aoi_geodf, year, month)
        scenes = sendRequest(serviceUrl + "scene-search", search_payload, apiKey)
        pd.json_normalize(scenes['results'])
        if len(scenes['results']) == 0:
            print("Month for scenes empty, skipping...")
            continue

        # Collect File IDs
        entityIds = [result['entityId'] for result in scenes['results'] if result['options']['bulk']]

        # Add to basket
        listId = f"{dataset}_{year}_{str(month)}_{socket.gethostname()}"
        scn_list_add_payload = {
            "listId": listId,
            'idField': 'entityId',
            "entityIds": entityIds,
            "datasetName": dataset
        }
        sendRequest(serviceUrl + "scene-list-add", scn_list_add_payload, apiKey)

        # Select URL download
        download_opt_payload = {
            "listId": listId,
            "datasetName": dataset,
        }
        products = sendRequest(serviceUrl + "download-options", download_opt_payload, apiKey)
        pd.json_normalize(products)

        # Collect File URLs based on product
        downloads = []
        if 'landsat_ot_c2_l2' in dataset:
            for product in products:
                if product["secondaryDownloads"]:
                    for secDownload in product["secondaryDownloads"]:
                        if secDownload["bulkAvailable"] and any(band in secDownload['displayId'] for band in bandNames):
                            downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
                        if secDownload['displayId'].endswith('_MTL.txt'):
                            downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
        elif 'srtm_v3' in dataset:
            for product in products:
                if product["bulkAvailable"] and product["entityId"] and product["id"]:
                    downloads.append({"entityId": product["entityId"], "productId": product["id"]})
        elif 'nlcd_collection_lndcov' in dataset:
            for product in products:
                if product["bulkAvailable"] and product["entityId"] and product["id"]:
                    downloads.append({"entityId": product["entityId"], "productId": product["id"]})
        download_req_payload = {
            "downloads": downloads,
            "label": listId
        }
        download_request_results = sendRequest(serviceUrl + "download-request", download_req_payload, apiKey)

        # Download Files Via URL
        if dataset == 'landsat_ot_c2_l2':
            results = download_request_results['availableDownloads']
            for result in results:
                runDownload(threads, result['url'])
        elif dataset == 'srtm_v3':
            results = download_request_results['preparingDownloads']
            for result in results:
                runDownload(threads, result['url'])
        else:
            results = download_request_results['preparingDownloads']
            for result in results:
                runDownload(threads, result['url'])
        for t in threads:
            t.join()
        for file in os.listdir(unprocessed_dir):
            if file.endswith('.tar'):
                try: 
                    extract_specific_files(unprocessed_dir + '/' + file, unprocessed_dir)
                except:
                    print(f'Error: Could not extract file {file}')

        # Clear Basket
        remove_scnlst_payload = {"listId": listId}
        sendRequest(serviceUrl + "scene-list-remove", remove_scnlst_payload, apiKey)

        # Re-project Raster Files
        for tif in os.listdir(unprocessed_dir):
            temp_path = ".temp"  # Temporary file path
            input_path = unprocessed_dir + '/' + tif
            with rasterio.open(input_path) as src:
                try:
                    color_map = src.colormap(1)
                except ValueError:
                    color_map = None
                with rioxarray.open_rasterio(input_path) as raster:
                    raster = raster.rio.reproject("EPSG:4326")
                    raster.rio.to_raster(temp_path, driver="GTiff")
            if color_map:
                with rasterio.open(temp_path, "r+") as dst:
                    dst.write_colormap(1, color_map)
            os.replace(temp_path, input_path)
            if os.path.exists(temp_path):
                os.remove(temp_path)
        # Move Unprocessed Files to Raw Folder
        for file in os.listdir(unprocessed_dir):
            if ".txt" in file or ".tif" in file or ".TIF" in file:
                if '1arc_v3' in file:
                    moveToRaw(file, 'DEM', f'{year}-01-01', city)
                    continue
                if 'Annual_NLCD' in file:
                    moveToRaw(file, 'Land_Cover', f'{year}-01-01', city)
                    continue
                date, band, coordinate = getMetaFromLandsatTIRs(file)
                if band in ['MTL', 'B10', 'B2', 'B3', 'B4', 'B5', 'B6', 'QA_PIXEL']:
                    moveToRaw(file, 'oli', date, city)
        print('Finished moving')

        #You only need 1 month for these as they are annual
        if dataset == 'nlcd_collection_lndcov' or dataset == 'srtm_v3':
            break

    #Save progress
    with open('./Logs/raw_progress.txt', "a") as file:
        file.write(str(city) + ":" + str(year) + ":" + dataset + "\n")
    print('progress written for', city, year, dataset)

Directory './Unprocessed' already exists.
Directory './Data' already exists.
Directory './RawClippedRasters' already exists.
Directory './Data/LST' already exists.
Directory './Data/NDVI' already exists.
Directory './Data/NDWI' already exists.
Directory './Data/Land_Cover' already exists.
Directory './Data/Albedo' already exists.
Directory './Data/DEM' already exists.
Directory './Data/labelLST' already exists.
Directory './RawClippedRasters/LST' already exists.
Directory './RawClippedRasters/NDVI' already exists.
Directory './RawClippedRasters/NDWI' already exists.
Directory './RawClippedRasters/Land_Cover' already exists.
Directory './RawClippedRasters/Albedo' already exists.
Directory './RawClippedRasters/DEM' already exists.
Directory './RawClippedRasters/labelLST' already exists.
Logging in...


Login Successful, API Key Received!


In [ ]:
datasets = ['landsat_ot_c2_l2', 'srtm_v3', 'nlcd_collection_lndcov']
years = [year for year in range(2014, 2015)]

# Load city footprints from Esri Living Atlas
shapefile_folder = "./Data/area_shp/"
cities = []
aoi_geodfs = []
for file in os.listdir(shapefile_folder):
    if file.endswith(".shp"):
        cities.append(file.replace('Polygon_', '').replace('.shp', ''))
        aoi_geodf = gpd.read_file(shapefile_folder + file)
        aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
        if aoi_geodf.empty:
            sys.exit("Error: Shapefile contains no data.")
        aoi_geodfs.append(aoi_geodf)
print("Shapefiles loaded successfully.")

for j, dataset in enumerate(datasets):
    for year in years:
        i = 0
        while i < len(cities):
            if i % 7 == 0:
                notifySelf(f'Starting on year {year} in dataset {dataset} as city {cities[i]}...')
            try:
                clear_folder(unprocessed_dir)
                assert len(os.listdir(unprocessed_dir)) == 0, "Unprocessed directory is not empty."
                city, aoi_geodf = cities[i], aoi_geodfs[i]
                if os.path.exists('./Logs/raw_progress.txt'):
                    with open('./Logs/raw_progress.txt', 'r') as file:
                        progress = [line.split(':') for line in file.read().strip().split('\n')]
                    if any(city == instance[0] and str(year) == instance[1] and dataset == instance[2] for instance in progress):
                        print(f"{city}, {year}, {dataset} was gathered in the past.")
                    else:
                        gatherRawRasters(dataset, year, city, aoi_geodf)
                i += 1
            except Exception as e:
                print("An exception occurred:")
                print(f"Exception: {e}")
                notifySelf("An exception occurred:")
                notifySelf(f"Exception: {e}")
                traceback.print_exc()  # Print the full stack trace
                time.sleep(15)
print("Gathered data successfully.")
notifySelf("Gathered data successfully.")

Shapefiles loaded successfully.
Deleted file: ./Unprocessed/LC08_L2SP_033033_20140305_20200912_02_T1_SR_B3.TIF
Deleted file: ./Unprocessed/LC08_L2SP_033033_20140305_20200912_02_T1_SR_B6.TIF
Deleted file: ./Unprocessed/LC08_L2SP_033033_20140305_20200912_02_T1_SR_B2.TIF
Deleted file: ./Unprocessed/LC08_L2SP_033033_20140305_20200912_02_T1_MTL.txt
Deleted file: ./Unprocessed/LC08_L2SP_033033_20140305_20200912_02_T1_SR_B5.TIF
Deleted file: ./Unprocessed/LC08_L2SP_033033_20140305_20200912_02_T1_SR_B4.TIF
InputData, Pahrump_NV, 2013, landsat_ot_c2_l2 was gathered in the past.
InputData, Savannah_GA, 2013, landsat_ot_c2_l2 was gathered in the past.
InputData, Palm_Coast_FL, 2013, landsat_ot_c2_l2 was gathered in the past.
InputData, Lexington-Fayette_KY, 2013, landsat_ot_c2_l2 was gathered in the past.
InputData, Casa_Grande_AZ, 2013, landsat_ot_c2_l2 was gathered in the past.
InputData, Little_Rock_AR, 2013, landsat_ot_c2_l2 was gathered in the past.
InputData, Buckeye_AZ, 2013, landsat_ot_c2

In [ ]:
'''Clip Raster, we are reading metadata from rastername'''
shapefile_folder = "./Data/area_shp/"
cities_aoi = {} # Use cities dictionary instead of list to read from the raster itself
for file in os.listdir(shapefile_folder):
    if file.endswith(".shp"):
        aoi_geodf = gpd.read_file(shapefile_folder + file)
        aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
        if aoi_geodf.empty:
            sys.exit("Error: Shapefile contains no data.")
        cities_aoi[file.replace('Polygon_', '').replace('.shp', '')] = aoi_geodf

#Get all tifs, copy everything else
notifySelf("Starting clip list of raster paths...")
allGeoFiles = get_file_paths(raw_dir)
usableTIFFs = []
for geoFilePath in tqdm(allGeoFiles, desc="Make raster list/move txt files"):
    geoFileParts = geoFilePath.split('/')
    fileName, date, city, dataType = geoFileParts[-1], geoFileParts[-2], geoFileParts[-3], geoFileParts[-4]
    targetFile = os.path.join(clipped_dir, dataType, city, date, fileName)
    if os.path.exists(targetFile):
        continue
    polygon = cities_aoi[city]
    if geoFilePath.endswith(".tif") or geoFilePath.endswith(".TIF"):
        if checkPolygonInRasterCompletely(polygon, geoFilePath):
            usableTIFFs.append(geoFilePath)
    elif geoFilePath.endswith(".txt"):
        moveToClipped(geoFilePath, fileName, dataType, date, city)